# In-silico credibility assessment of a computational physiological model for non-invasive monitoring of respiratory effort in critically ill patients

#TODO: Add explanation here

In [ ]:
# For exact reproducibility, package versions are specified.
# numpy == 2.2.2
# scipy == 1.15.1
# statsmodels == 0.14.4

# 1. Import the required libraries

In [ ]:
# Standard code libraries
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Custom code libraries from ReSurfEMG
from resurfemg.data_connector.config import Config
from resurfemg.pipelines import ipy_widgets
from resurfemg.data_connector import file_discovery
from resurfemg.data_connector.converter_functions import load_file
from resurfemg.preprocessing.ecg_removal import detect_ecg_peaks, gating
from resurfemg.preprocessing import filtering as filt
from resurfemg.preprocessing import pneumatic as pneu
from resurfemg.postprocessing import event_detection as evt
from resurfemg.postprocessing import features as feat
from resurfemg.postprocessing import quality_assessment as qa

from resurfemg.data_connector.data_classes import (
    VentilatorDataGroup, EmgDataGroup)

from resurfemg.modelling import tau_estimation as tau_est
from resurfemg.modelling import fit
from resurfemg.modelling import integrated_equation_of_motion as ieqm

warnings.filterwarnings('ignore')

%matplotlib widget

## 2. Load the ventilator and sEMG data

In [ ]:
# Identify all recordings available for the selected patient profile

# First find the patients
config = Config(verbose=False)

# Then find the files for the selected patients:
base_path = config.get_directory('vvuq_iEqM_baseline')
folder_levels = None

emg_files = file_discovery.find_files(
    base_path=base_path,
    file_name_regex='y_emg_baseline',
    extension_regex='*',
    folder_levels=folder_levels)

vent_files = file_discovery.find_files(
    base_path=base_path,
    file_name_regex='y_vent_baseline',
    extension_regex='*',
    folder_levels=folder_levels)
folder_levels = ['files']

In [ ]:
# If you want to select another file:
btn_list_emg = ipy_widgets.file_select(
    emg_files,
    folder_levels=folder_levels,
    default_value_select=None,
    default_idx_select=None)
btn_list_vent = ipy_widgets.file_select(
    vent_files,
    folder_levels=folder_levels,
    default_value_select=None,
    default_idx_select=None)

In [ ]:
# Load the EMG and ventilator data recordings from the selected folders.
emg_file_chosen = os.path.join(base_path, *[btn.value for btn in btn_list_emg])
vent_file_chosen = os.path.join(base_path, *[btn.value for btn in btn_list_vent])

# Store the EMG data in a group of TimeSeries objects
print('--- Loading EMG data ---')
y_emg, _, metadata_emg = load_file(emg_file_chosen, verbose=False)
metadata_emg['fs'] = 2048  # Hz
metadata_emg['labels'] = ['ECG', 'sEMGmus']
metadata_emg['units'] = ['uV', 'uV']
emg_timeseries = EmgDataGroup(
    y_emg,
    fs=metadata_emg['fs'],
    labels=metadata_emg['labels'],
    units=metadata_emg['units'] )

# Store the ventilator data in a group of TimeSeries objects
print('\n--- Loading ventilator data ---')
y_vent, _, metadata_vent = load_file(vent_file_chosen, verbose=False)
metadata_vent['fs'] = 100  # Hz
metadata_vent['labels'] = ['Paw', "V'", 'V']
metadata_vent['units'] = ['cmH2O', 'L/s', 'L']
vent_timeseries = VentilatorDataGroup(
    y_vent,
    fs=metadata_vent['fs'],
    labels=metadata_vent['labels'],
    units=metadata_vent['units'])

emg_mus = emg_timeseries[1]

# 3. Pre-process the data

## 3.a EMG processing

In [ ]:
# Filter
emg_mus.filter_emg(signal_io=('raw', 'filt'), hp_cf=1, lp_cf=500, order=3)
emg_mus.filter_emg(signal_io=('filt', 'filt'), hp_cf=20, lp_cf=500, order=3)

In [ ]:
# ECG peak detection:
ecg_peak_idxs = detect_ecg_peaks(
    ecg_raw=emg_timeseries[0]['raw'],
    fs=metadata_emg['fs'],
    peak_fraction=0.50,
    peak_width_s=int(0.001 * metadata_emg['fs']),
    peak_distance=int(metadata_emg['fs'] / 3),
)
emg_timeseries.run(
    'set_peaks',
    peak_idxs=ecg_peak_idxs,
    signal=emg_timeseries[0]['raw'],
    peak_set_name='ecg',
    overwrite=True
)

# ECG removal through gating
emg_mus['clean'] = gating(
    emg_raw=emg_mus['filt'],
    peak_idxs=ecg_peak_idxs,
    gate_width=int(0.20 * metadata_emg['fs']),
    method=3,
)

In [ ]:
# Calculate the RMS envelope of the signal
emg_mus.envelope(
    env_window=int(metadata_emg['fs'] * 250 / 1000)
)

In [ ]:
# Calculate the baseline for the EMG envelopes and p_vent
emg_mus.baseline(
    percentile=33,
    window_s=int(7.5 * emg_timeseries.param['fs']),
    step_s=int(emg_timeseries.param['fs'] / 5)
)
vent_timeseries.run(
    'baseline',
    channel_idxs=[0],
    window_s=int(5.0 * vent_timeseries.param['fs']),
    step_s=int(vent_timeseries.param['fs'] / 5),
    signal_io=('raw', 'baseline')
)

## 3.b Ventilator data processing

In [ ]:
#  Detect end-expiratory samples: zero crossing flow 
flow_vent = vent_timeseries[1]
v_vent = vent_timeseries[2]
zc, _ = pneu.zero_cros_flow(flow_vent['raw'], flow_threshold=0.2)

# Calculate the volume from the flow signal
v_vent['clean'], _, _, _ = pneu.volume_computation(
    t=flow_vent.t_data,
    flow=flow_vent['raw'],
    fs=flow_vent.param['fs'],
    zc=zc,
    method="Last points"
)

## 3.c Plot resulting data

In [ ]:
# Plot the raw data with the envelope
# EMG data
n_rows = max([len(emg_timeseries.channels), len(vent_timeseries.channels)])
fig, axis = plt.subplots(nrows=n_rows, ncols=2, figsize=(12, n_rows*2), sharex=True)
axes_emg = axis[:len(emg_timeseries.channels), 0]
colors = ['tab:cyan', 'tab:orange']
emg_timeseries[0].plot_full(
    axes=axes_emg[0],
    signal_io=('raw',),
    baseline_bool=False)
emg_timeseries[1].plot_full(
    axes=axes_emg[1],
    signal_io=('clean',),
    baseline_bool=False)
emg_timeseries[1].plot_full(
    axes=axes_emg[1], signal_io=('env',), colors=colors)

axes_emg[0].set_title('EMG data')
axes_emg[-1].set_xlabel('t (s)')

# Ventilator data data
axes_vent = axis[:, 1]
vent_timeseries.run('plot_full', axes=axes_vent)
axes_vent[0].set_title('Ventilator data')
axes_vent[-1].set_xlabel('t (s)')

if n_rows > len(emg_timeseries.channels):
    for ax_idx in range(len(emg_timeseries.channels), n_rows):
        axis[ax_idx, 0].axis('off')
    axis[len(emg_timeseries.channels)-1, 0].tick_params(
        axis='x', which='both', labelbottom=True)

axes_emg[-1].set_xlim([0, 10])

# 4. Identify supported breaths in p_vent

In [ ]:
# Set p_vent_idx if it is not already set
if vent_timeseries.p_vent_idx is None:
    vent_timeseries.p_vent_idx = 0

# Set v_vent_idx if it is not already set
if vent_timeseries.v_vent_idx is None:
    vent_timeseries.v_vent_idx = 2

In [ ]:
# Detect PEEP
vent_timeseries.find_peep(pressure_idx=0, volume_idx=2)

# Find supported breath pressures
p_vent = vent_timeseries[vent_timeseries.p_vent_idx]
f_vent = vent_timeseries[1]
v_vent = vent_timeseries[vent_timeseries.v_vent_idx]

# Find the peaks in Paw
vent_timeseries.find_ventilator_peaks(channel_io=(0, 0), overwrite=True)
# Find the peaks in Volume
vent_timeseries.find_ventilator_peaks(channel_io=(2, 2), overwrite=True)

paw_peaks = p_vent.peaks['ventilator_breaths'].peak_df['peak_idx']
ps_set = np.round(np.median(p_vent['raw'][paw_peaks] - vent_timeseries.peep))

print(f"Number of Paw peaks: {len(paw_peaks)}")
print(f"PEEP set: {vent_timeseries.peep} cmH2O")
print(f"PS set: {ps_set}")


# 4. Tau estimation

In [ ]:
# Create a mask to select the flow and volume samples for tau estimation
mask_params = {
    'theta_paw_peep': 0.5,
    'min_duration': 1.5,
    'min_tv': 0.125,
    'max_v0': 0.1,
    'min_vol': -0.05,
}

tau_breath_df, tau_mask, tau_submasks_df = tau_est.tau_mask(
    p_aw=p_vent['raw'],
    flow=f_vent['raw'],
    volume=v_vent['clean'],
    peep=vent_timeseries.peep,
    zc=zc,
    **mask_params
)

In [ ]:
# Plot selected samples for tau estimation
fig, ax = plt.subplots(figsize=(5, 4)) 
ax.plot(tau_breath_df.flow[tau_mask], tau_breath_df.volume[tau_mask], "o",
        color='tab:green', label='Selected samples')
ax.plot(tau_breath_df.flow, tau_breath_df.volume,
        color='tab:blue', label='All samples')
ax.set_xlabel("V' (L/s)")
ax.set_ylabel("V (L)")

In [ ]:
# Estimate tau using a robust linear model with Tukey's biweight function
tau, mdl_tau = tau_est.tau_switch_smf(
    tau_breath_df[tau_mask], verbose=False)
mdl_tau_offset = mdl_tau.params[:-1]

In [ ]:
# Plot tau estimation results
fig, ax = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(8, 4)) 
ax[0].plot(tau_breath_df.flow[tau_mask], tau_breath_df.volume[tau_mask], "o",
        color='tab:green', label='Selected samples')
ax[0].plot(tau_breath_df.flow, tau_breath_df.volume,
        color='tab:blue', label='All samples')
ax[0].set_xlabel("V' (L/s)")
ax[0].set_ylabel("V (L)")
ax[0].set_title("Flow-Volume curve")
ax[0].legend(loc="upper right")
ax[0].grid()

ax[1].plot(tau_breath_df.flow[tau_mask], tau_breath_df.volume[tau_mask], "o",
           color='tab:blue', label='Selected samples')
ax[1].plot(
    tau_breath_df.flow[tau_mask],
    -tau * tau_breath_df.flow[tau_mask] + mdl_tau_offset.iloc[0],
    ".-", color='tab:orange', linewidth=4,
    label=f"Fitted RLM tau={tau:.3f}",
)
ax[1].set_xlabel("V' (L/s)")
ax[1].set_ylabel("V (L)")
ax[1].set_title("Tau estimation")
ax[1].legend(loc="upper right")
ax[1].grid()
fig.tight_layout(pad=1.0)

In [ ]:
# Model: Tau estimation: fit
y_pred_tau = mdl_tau.predict(
    tau_breath_df.loc[tau_mask, ["flow", "breath_id"]])
mask_nan = ~np.isnan(y_pred_tau)

r_true = 15
c_true = 0.05
tau_true = r_true * c_true
tau_error = (tau - tau_true) / tau_true * 100

rse_tau = fit.residual_standard_error(
    tau_breath_df.loc[tau_mask, "volume"].values[mask_nan],
    y_pred_tau.values[mask_nan], 1
)
cod_tau = fit.explained_variance(
    tau_breath_df.loc[tau_mask, "volume"].values[mask_nan],
    y_pred_tau.values[mask_nan]
)
r2_tau = np.corrcoef(
    tau_breath_df.loc[tau_mask, "volume"].values[mask_nan],
    y_pred_tau.values[mask_nan]
)[0, 0]**2
explained_variance_tau = fit.explained_variance(
    tau_breath_df.loc[tau_mask, "volume"].values[mask_nan],
    y_pred_tau.values[mask_nan]
)

print(f"Tau Error: {tau_error:.1f}%")
print(f"RSE: {rse_tau}")
print(f"Coefficient of Determination (R²): {cod_tau}")
print(f"Pearson R²: {r2_tau}")

# 5. Vrs calculation

In [ ]:
# Estimate Vrs for the integrated equation of motion
v_rs_signal = ieqm.v_rs_estimation(
    volume=v_vent['clean'],
    flow=f_vent['raw'],
    tau=tau,
    expiration_mask=tau_submasks_df['expiration'].to_numpy(),
    window_s=int(5.0 * vent_timeseries.param['fs']),
    step_s=int(vent_timeseries.param['fs'] / 5),
    set_percentile=33,
    omit_nan=True
)
v_rs_vent_group = VentilatorDataGroup(
    v_rs_signal,
    fs=vent_timeseries.param['fs'],
    labels=['Vrs'],
    units=['L']
)
v_rs = v_rs_vent_group[0]
v_rs.baseline(
    signal_io=('raw', 'baseline'),
    percentile=33,
    window_s=int(5.0 * vent_timeseries.param['fs']),
    step_s=int(vent_timeseries.param['fs'] / 5)
)

In [ ]:
# Plot Paw, Vrs, and sEAmus

fig, ax = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(8, 6))

p_vent.plot_full(axes=ax[0], signal_io=('raw',), baseline_bool=True)
p_lim = ax[0].get_ylim()
ax[0].set_ylim(0, p_lim[1] + 1)
v_rs.plot_full(axes=ax[1], signal_io=('raw',), baseline_bool=True)
emg_mus.plot_full(axes=ax[2], signal_io=('env',), baseline_bool=True)
ax[-1].set_xlabel("t (s)")

ax[-1].set_xlim([0, 20])

# 6. Calculate features from selected signals

## 6.a. EMG features

In [ ]:
# Find sEAmus peaks
# Detect peaks
emg_mus.detect_emg_breaths(
    peak_set_name='breaths',
    threshold=0,
    min_peak_width_s=1,
    prominence_factor=0.7)

# Detect onset and offset of the detected peaks
emg_mus.peaks['breaths'].detect_on_offset(
    baseline=emg_mus['baseline']
)

In [ ]:
# Calculate ETPmus
emg_mus.calculate_time_products(
    peak_set_name='breaths', parameter_name='ETPmus')

In [ ]:
# Test EMG quality on an individual peak basis. This includes the following
# tests:
# - interpeak_distance: EMG interpeak time as compared to ECG interbeat time
# - snr: pseudo signal-to-noise ratio
# - aub: area under the baseline relative to total ETP
# - curve_fit: deviation of the bell curve fit relative to total ETP
#
# Relative ETP and AUB are tested during a second stage, including only peaks
# that passed the first stage.

parameter_names = {
    'time_product': 'ETPmus'
}
emg_mus.test_emg_quality(
    skip_tests=['relative_etp', 'relative_aub'],
    peak_set_name='breaths',
    parameter_names=parameter_names,
    verbose=False)

In [ ]:
# Remove peaks that did not pass the quality tests
emg_mus.peaks['breaths'].sanitize()

In [ ]:
# Test EMG quality for relative_etp and relative_aub
parameter_names = {
    'time_product': 'ETPmus'
}
emg_mus.test_emg_quality(
    skip_tests=[],
    peak_set_name='breaths',
    parameter_names=parameter_names,
    verbose=False)

In [ ]:
# Remove peaks that did not pass the quality tests
emg_mus.peaks['breaths'].sanitize()

In [ ]:
# TODO: SUMMARY OF QUALITY OUTCOMES
# qual_cols = ['snr', 'aub', 'bell', 'relative_etp', 'relative_aub']
# qual_cols_name = ['SNR', 'AUB %', 'Bell', 'ETP', 'AUB']
# summary_df = emg_mus.peaks['breaths'].quality_outcomes_df[qual_cols]
# summary_df.columns = qual_cols_name
# count_df = np.sum(summary_df, axis=0)
# count_df, summary_df.shape[0]

## 6.b. PTPaw, PTPpeep

In [ ]:
# Detect supported breaths in Paw
# Peak detection was done at 3. for PEEP and PS detection

# Detect onset and offset of the detected peaks
p_vent.peaks['ventilator_breaths'].detect_on_offset(
    baseline=p_vent['baseline'], 
)
p_vent.peaks['ventilator_breaths'].sanitize()

In [ ]:
# Calculate PTPaw, PTPpeep, and PTPaw-PTPpeep
# Calculate PTPaw between Paw and zero
p_vent['zero'] = np.zeros(p_vent['baseline'].shape)
p_vent.calculate_time_products(
    peak_set_name='ventilator_breaths',
    parameter_name='PTPaw',
    signal_io=('zero',),
    include_aub=False)
# Calculate PTPpeep between PEEP and zero
p_vent['peep'] = p_vent['zero'] + vent_timeseries.peep
p_vent.calculate_time_products(
    peak_set_name='ventilator_breaths',
    parameter_name='PTPaw-PTPpeep',
    signal_io=('peep',),
    include_aub=False)
# Calculate PTPpeep as the difference between PTPaw and PTPaw-PTPpeep
paw_vent_peak_df = p_vent.peaks['ventilator_breaths'].peak_df
paw_vent_peak_df['PTPpeep'] = (paw_vent_peak_df['PTPaw']
                               - paw_vent_peak_df['PTPaw-PTPpeep'])

## 6.c. VTPrs

In [ ]:
# Detect supported breaths in Vrs
# Peak detection
v_rs_vent_group.find_ventilator_peaks(
    channel_io=(0, 0),
    peak_set_name='breaths',
        width_s=int(0.2 * v_rs.param['fs']),
        threshold=0,
        prominence=0,
        threshold_new=0,
        prominence_new=0.2,
        overwrite=True
)

# Detect onset and offset of the detected peaks
v_rs.peaks['breaths'].detect_on_offset(
    baseline=v_rs['baseline'],
)
v_rs.peaks['breaths'].sanitize()

In [ ]:
# Calculate VTPrs
v_rs['zero'] = np.zeros(v_rs['baseline'].shape)
v_rs.calculate_time_products(
    peak_set_name='breaths',
    parameter_name='VTPrs',
    signal_io=('zero',),
    include_aub=False)


# 7. Link sEAmus breaths to closest Paw and Vrs peak

In [ ]:
# Match each peak in sEAmus to closest peaks in Paw and Vrs with a
# max_time_diff tolerance. If no match is found, returns None for that peak.
t_peaks_emg = emg_mus.peaks['breaths']['peak_idx'] / emg_mus.param['fs']
t_peaks_p_aw = p_vent.peaks['ventilator_breaths']['peak_idx'] / p_vent.param['fs']
t_peaks_v_rs = v_rs.peaks['breaths']['peak_idx'] / v_rs.param['fs']

max_time_diff = 1.0  # seconds
matched_idxs = []
for idx1, t_peak_emg in enumerate(t_peaks_emg):
    # Signal 2
    if len(t_peaks_p_aw) > 0:
        diffs2 = np.abs(t_peaks_p_aw - t_peak_emg)
        min_diff2_idx = np.argmin(diffs2)
        idx2 = min_diff2_idx if diffs2[min_diff2_idx] <= max_time_diff else None
    else:
        idx2 = None

    # Signal 3
    if len(t_peaks_v_rs) > 0:
        diffs3 = np.abs(t_peaks_v_rs - t_peak_emg)
        min_diff3_idx = np.argmin(diffs3)
        idx3 = min_diff3_idx if diffs3[min_diff3_idx] <= max_time_diff else None
    else:
        idx3 = None

    matched_idxs.append((idx1, idx2, idx3))

# Create a DataFrame to store the matched indices and drop any rows with NaN values
df_linked_peaks = pd.DataFrame(
    matched_idxs, columns=["sEAmus_idx", "Paw_idx", "Vrs_idx"])
df_linked_peaks = df_linked_peaks.dropna(axis=0)

emg_sel_idxs = df_linked_peaks["sEAmus_idx"].astype(int).to_numpy()
paw_sel_idxs = df_linked_peaks["Paw_idx"].astype(int).to_numpy()
vrs_sel_idxs = df_linked_peaks["Vrs_idx"].astype(int).to_numpy()

# Create a DataFrame to store the selected parameters for the linked peaks
df_ieqm = pd.DataFrame({
    "ETPmus": emg_mus.peaks['breaths'].peak_df['ETPmus'].iloc[emg_sel_idxs].values,
    "PTPaw": p_vent.peaks['ventilator_breaths'].peak_df['PTPaw'].iloc[paw_sel_idxs].values,
    "PTPpeep": p_vent.peaks['ventilator_breaths'].peak_df['PTPpeep'].iloc[paw_sel_idxs].values,
    "PTPaw-PTPpeep": p_vent.peaks['ventilator_breaths'].peak_df['PTPaw-PTPpeep'].iloc[paw_sel_idxs].values,
    "VTPrs": v_rs.peaks['breaths'].peak_df['VTPrs'].iloc[vrs_sel_idxs].values
})

In [ ]:
# TODO: Plot VTPrs, PTPaw, ETPrs for 3D evaluation


# 8. Fit integrated Equation of Motion

In [ ]:
# Estimate compliance and NMC using the integrated equation of motion
c_true = 50.0  # mL/cmH2O
nmc_true = 3.0  # cmH2O/uV

keys = {'PTP': 'PTPaw-PTPpeep', 'VTP': 'VTPrs', 'ETP': 'ETPmus'}
c_est, nmc_est, mdl_ieqm = ieqm.compliance_estimation(
    df_ieqm, t=1.345, keys=keys, verbose=False)
r_est = tau / c_est

print("Model results:")
print(f"The compliance is: {c_est:.2f} mL/cmH2O")
print(f"The compliance error is: {(np.abs(c_est) - c_true)/c_true * 100:.1f} %\n")
print(f"NMC is: {nmc_est:.2f} cmH2O/uV")
print(f"NMC error is: {(np.abs(nmc_est) - nmc_true) / nmc_true * 100:.1f} %\n")


# 9. Compute Pmus and PTPmus

## 9.a. Load true Pmus data

In [ ]:
# Find all p_mus files in the base path
folder_levels = None

p_mus_files = file_discovery.find_files(
    base_path=base_path,
    file_name_regex='y_p_mus_baseline',
    extension_regex='*',
    folder_levels=folder_levels)

folder_levels = ['files']

In [ ]:
# If you want to select another file:
btn_list_p_mus = ipy_widgets.file_select(
    p_mus_files,
    folder_levels=folder_levels,
    default_value_select=None,
    default_idx_select=None)

In [ ]:
# Load the Pmus data recordings from the selected folders.
p_mus_file_chosen = os.path.join(base_path, *[btn.value for btn in btn_list_p_mus])

# Store the Pmus data in a group of TimeSeries objects
print('--- Loading Pmus data ---')
y_p_mus, _, metadata_p_mus = load_file(p_mus_file_chosen, verbose=False)
metadata_p_mus['fs'] = 2048  # Hz
metadata_p_mus['labels'] = ['Pmus']
metadata_p_mus['units'] = ['cmH2O']
p_mus_timeseries = VentilatorDataGroup(
    y_p_mus,
    fs=metadata_p_mus['fs'],
    labels=metadata_p_mus['labels'],
    units=metadata_p_mus['units'] )

p_mus_true = p_mus_timeseries[0]

## 9.b Calculate estimated Pmus

In [ ]:
# Calculate the predicted Pmus using the estimated compliance and NMC
y_p_mus_est = ieqm.p_mus_estimation(
    p_aw=p_vent['raw'],
    peep=vent_timeseries.peep,
    v_rs=v_rs['raw'],
    c=c_est
)
p_mus_est_group = VentilatorDataGroup(
    y_p_mus_est,
    fs=vent_timeseries.param['fs'],
    labels=['Pmus,est'],
    units=['cmH2O']
)
p_mus_est = p_mus_est_group[0]

In [ ]:
# Filter the estimated Pmus signal according to Graßhoff 2023
p_mus_est['filt'] = filt.emg_lowpass_butter(
    p_mus_est['raw'],
    low_pass=3.5,
    fs_emg=p_mus_est.param['fs'],
    order=7
)

In [ ]:
# TODO: Plot Pmus,true and Pmus,est

## 9.c. Calculate PTPmus,true and PTPmus,est

In [ ]:
# Calculate baselines
# Pmus,true baseline
p_mus_true.baseline(
    signal_io=('raw', 'baseline'),
    percentile=33,
    window_s=int(5.0 * p_mus_true.param['fs']),
    step_s=int(p_mus_true.param['fs'] / 5)
)
# Pmus,est baseline
p_mus_est.baseline(
    signal_io=('filt', 'baseline'),
    percentile=33,
    window_s=int(5.0 * p_mus_est.param['fs']),
    step_s=int(p_mus_est.param['fs'] / 5)
)

In [ ]:
# Detect peaks in the Pmus,true and Pmus,est signals
# Pmus,true
peak_idxs_true = find_peaks(p_mus_true['raw'], prominence=0.5, height=0.2)[0]
p_mus_true.set_peaks(
    peak_set_name='breaths',
    peak_idxs=peak_idxs_true,
    signal=p_mus_true['raw']
)

# Pmus,est
peak_idxs_est = find_peaks(p_mus_est['filt'], prominence=0.5, height=0.2)[0]
p_mus_est.set_peaks(
    peak_set_name='breaths',
    peak_idxs=peak_idxs_est,
    signal=p_mus_est['filt']
)

## 9.d. Link Pmus,est and Pmus,true breaths

In [ ]:
# Match each peak in Pmus,true to closest peaks in Pmus,est with a
# max_time_diff tolerance. If no match is found, returns None for that peak.
t_peaks_true = p_mus_true.peaks['breaths']['peak_idx'] / p_mus_true.param['fs']
t_peaks_est = p_mus_est.peaks['breaths']['peak_idx'] / p_mus_est.param['fs']

max_time_diff = 2.0  # seconds
matched_idxs_p_mus = []
for idx1, t_peak_true in enumerate(t_peaks_true):
    # Signal 2
    if len(t_peaks_est) > 0:
        diffs2 = np.abs(t_peaks_est - t_peak_true)
        min_diff2_idx = np.argmin(diffs2)
        idx2 = min_diff2_idx if diffs2[min_diff2_idx] <= max_time_diff else None
    else:
        idx2 = None

    matched_idxs_p_mus.append((idx1, idx2))

# Create a DataFrame to store the matched indices and drop any rows with NaN values
df_linked_peaks_pmus = pd.DataFrame(
    matched_idxs_p_mus, columns=["Pmus,true_idx", "Pmus,est_idx"])
df_linked_peaks_pmus = df_linked_peaks_pmus.dropna(axis=0)

true_sel_idxs = df_linked_peaks_pmus["Pmus,true_idx"].astype(int).to_numpy()
est_sel_idxs = df_linked_peaks_pmus["Pmus,est_idx"].astype(int).to_numpy()

p_mus_true_idxs_sel = p_mus_true.peaks['breaths']['peak_idx'][true_sel_idxs]
p_mus_est_idxs_sel = p_mus_est.peaks['breaths']['peak_idx'][est_sel_idxs]

In [ ]:
# Detect on- and offsets in the linked Pmus,true and Pmus,est peaks
# Pmus,true
p_mus_true.set_peaks(
    peak_set_name='breaths_sel',
    peak_idxs=p_mus_true_idxs_sel,
    signal=p_mus_true['raw']
)
p_mus_true.peaks['breaths_sel'].detect_on_offset(
    baseline=p_mus_true['baseline'],
)

# Pmus,est
p_mus_est.set_peaks(
    peak_set_name='breaths_sel',
    peak_idxs=p_mus_est_idxs_sel,
    signal=p_mus_est['filt']
)
p_mus_est.peaks['breaths_sel'].detect_on_offset(
    baseline=p_mus_est['baseline'],
)

In [ ]:
# Calculate PTPmus,true and PTPmus,est
# Pmus,true
p_mus_true.calculate_time_products(
    peak_set_name='breaths_sel',
    parameter_name='PTPmus,true',
    signal_io=('baseline',),
    include_aub=False)
# Pmus,est
p_mus_est.calculate_time_products(
    peak_set_name='breaths_sel',
    parameter_name='PTPmus,est',
    signal_io=('baseline',),
    include_aub=False)

In [ ]:
# Evaluate model performance by comparing PTPmus,true and PTPmus,est
ptp_true = p_mus_true.peaks['breaths_sel'].peak_df['PTPmus,true'].to_numpy()
ptp_est = p_mus_est.peaks['breaths_sel'].peak_df['PTPmus,est'].to_numpy()

diff = ptp_est - ptp_true
diff_perc = (diff / ptp_true) * 100

print(f"The PTPmus mean deviation is: {np.mean(diff):.2f} cmH2O*s")
print(f"The PTPmus percentage deviation is: {np.mean(diff_perc):.1f} %")

In [ ]:
# Plot residuals
fig, ax = plt.subplots(nrows=1, ncols=1, sharey=True, figsize=(4, 3))
fig.tight_layout(pad=1.5)
ax.axhline(0, color="r", linestyle="--")
ax.plot(ptp_true, diff, "x", color='tab:blue')
ax.set_xlabel('PTPmus,true (cmH2O*s)')
ax.set_ylabel('PTPmus,est - PTPmus,true (cmH2O.s)')
ax.set_title('Residuals of PTPmus estimation')

# 10. Compute model performance indicators

## 10.a. R²(NMC)

In [ ]:
# Match each peak in Pmus,est to closest peaks in sEAmus and Vvent with a
# max_time_diff tolerance. If no match is found for all three signals,
# that peak is discarded.

t_peaks_p_mus_est = p_mus_est.peaks['breaths_sel']['peak_idx'] / p_mus_est.param['fs']
t_peaks_emg = emg_mus.peaks['breaths']['peak_idx'] / emg_mus.param['fs']
t_peak_v_vent = v_vent.peaks['ventilator_breaths']['peak_idx'] / v_vent.param['fs']

max_time_diff = 1.5  # seconds
matched_idxs = []
for idx1, t_peak_p_mus_est in enumerate(t_peaks_p_mus_est):
    # sEAmus
    if len(t_peaks_emg) > 0:
        diffs2 = np.abs(t_peak_p_mus_est - t_peaks_emg)
        min_diff2_idx = np.argmin(diffs2)
        idx2 = min_diff2_idx if diffs2[min_diff2_idx] <= max_time_diff else None
    else:
        idx2 = None

    # Vvent
    if len(t_peak_v_vent) > 0:
        diffs3 = np.abs(t_peak_p_mus_est - t_peak_v_vent)
        min_diff3_idx = np.argmin(diffs3)
        idx3 = min_diff3_idx if diffs3[min_diff3_idx] <= max_time_diff else None
    else:
        idx3 = None

    matched_idxs.append((idx1, idx2, idx3))

# Store the matched indices in a DataFrame and drop any rows with NaN values
df_linked_peaks = pd.DataFrame(
    matched_idxs, columns=["p_mus_est_nr", "sEAmus_nr", "v_vent_nr"])
df_linked_peaks = df_linked_peaks.dropna(axis=0)

# Extract the selected peak indices for Pmus,est and sEAmus
p_mus_est_nrs = df_linked_peaks["p_mus_est_nr"].astype(int).to_numpy()
sEAmus_nrs = df_linked_peaks["sEAmus_nr"].astype(int).to_numpy()

In [ ]:
# Calculate R² between PTPmus,est and ETPmus for the linked peaks
ptp_mus_est = p_mus_est.peaks['breaths_sel'].peak_df['PTPmus,est'].to_numpy()[p_mus_est_nrs]
etp_mus = emg_mus.peaks['breaths'].peak_df['ETPmus'].to_numpy()[sEAmus_nrs]

r2_nmc = np.corrcoef(ptp_mus_est, etp_mus)[0, 1] ** 2
print(f"The R² between PTPmus,est and ETPmus is: {r2_nmc * 100:.1f}%")

## 10.b. SNR

In [ ]:
# Calculate the SNR of the sEAdi signal using the percentile SNR method
# See Graßhoff et al., Crit. Care, 2021 for details on the method.

snr = feat.percentile_snr(
    env=emg_mus['env'],
    window_length=int(10 * emg_mus.param['fs']))

print(f"The SNR of the sEAdi signal is: {snr:.1f}")

## 10.c. ΔTei,neural

In [ ]:
# Find the onset of inspiratory flow matched to the ventilator support breaths
t_zcs = zc / vent_timeseries.param['fs']
t_p_aws = p_vent.peaks['ventilator_breaths']['peak_idx'] / p_vent.param['fs']

max_time_diff = 1.5  # seconds
matched_idxs_zc = []
for idx1, t_zc in enumerate(t_zcs):
    # Paw
    if len(t_p_aws) > 0:
        diffs2 = np.abs(t_p_aws - t_zc)
        min_diff2_idx = np.argmin(diffs2)
        idx2 = min_diff2_idx if diffs2[min_diff2_idx] <= max_time_diff else None
    else:
        idx2 = None

    matched_idxs_zc.append((idx1, idx2))

# Store the matched indices in a DataFrame and drop any rows with NaN values
df_linked_peaks_pmus = pd.DataFrame(
    matched_idxs_zc, columns=["zc_idx", "p_aw_idx"])
df_linked_peaks_pmus = df_linked_peaks_pmus.dropna(axis=0)

zc_sel_idxs = zc[df_linked_peaks_pmus["zc_idx"].astype(int).to_numpy()]


In [ ]:
# Match each peak in flow onset (zc) to closest peaks in Paw and Vrs with a
# max_time_diff tolerance. If no match is found for all three signals, that
# peak is discarded.

t_zcs_sel = zc_sel_idxs / vent_timeseries.param['fs']
t_peaks_p_mus_true = p_mus_true.peaks['breaths_sel']['peak_idx'] / p_mus_true.param['fs']
t_peak_v_vent = v_vent.peaks['ventilator_breaths']['peak_idx'] / v_vent.param['fs']

max_time_diff = 1.0  # seconds
matched_idxs = []
for idx1, t_zc_sel in enumerate(t_zcs_sel):
    # Signal 2
    if len(t_peaks_p_mus_true) > 0:
        diffs2 = np.abs(t_peaks_p_mus_true - t_zc_sel)
        min_diff2_idx = np.argmin(diffs2)
        idx2 = min_diff2_idx if diffs2[min_diff2_idx] <= max_time_diff else None
    else:
        idx2 = None

    # Signal 3
    if len(t_peaks_v_rs) > 0:
        diffs3 = np.abs(t_peaks_v_rs - t_peak_v_vent)
        min_diff3_idx = np.argmin(diffs3)
        idx3 = min_diff3_idx if diffs3[min_diff3_idx] <= max_time_diff else None
    else:
        idx3 = None

    matched_idxs.append((idx1, idx2, idx3))

# Store the matched indices in a DataFrame and drop any rows with NaN values
df_linked_peaks = pd.DataFrame(
    matched_idxs, columns=["zc_nr", "p_mus_true_nr", "v_vent_nr"])
df_linked_peaks = df_linked_peaks.dropna(axis=0)

# Extract the selected peak indices as numpy arrays
zc_nrs = df_linked_peaks["zc_nr"].astype(int).to_numpy()
p_mus_true_nrs = df_linked_peaks["p_mus_true_nr"].astype(int).to_numpy()

In [ ]:
# Find the onset of mechanical expiration
mechanical_exp_idx = np.zeros(zc[zc_nrs].shape, dtype=int)
for i, zc_i in enumerate(zc[zc_nrs] + 2):
    if i + 1 < len(zc[zc_nrs]):
        end_idx = zc[i + 1]
    else:
        end_idx = len(f_vent['raw'])

    _flow = f_vent['raw'][zc_i:end_idx]
    mechanical_exp_idx[i]= np.where(np.diff(np.sign(_flow)))[0][0] + zc_i

In [ ]:
# ΔTei,neural
ei_neural_idxs = evt.neural_expiratory_time(
    env=p_mus_true['raw'],
    peak_idxs=p_mus_true.peaks['breaths_sel']['peak_idx'][p_mus_true_nrs],
    end_idxs=p_mus_true.peaks['breaths_sel']['end_idx'][p_mus_true_nrs],
    threshold=0.70
)
t_ei_neural = ei_neural_idxs / p_mus_true.param['fs']
t_ei_mechanical = mechanical_exp_idx / vent_timeseries.param['fs']
dt_ei_neural = t_ei_mechanical - t_ei_neural

print(f"The mean ΔTei,neural is: {1000*np.mean(dt_ei_neural):.0f} ms")